# Similarity matrix for the expanded catalog

Same pipeline as `3_similarity.ipynb`, run on `data/expanded_catalog.json` (349 products, created in `1b_expanded_catalog.ipynb`). The original files are not touched; everything is written with an `_expanded` suffix.

1. Preprocess the expanded catalog into one record per unique English feature title (same rules as `1_data_overview_and_preprocessing.ipynb`).
2. Compute the similarity matrix with the identical method and weights as `3_similarity.ipynb`: 50% semantic (Qwen embeddings), 25% TF-IDF, 25% fuzzy title matching, minus the winter/windproof conflict penalty.
3. Save it to `data/similarity_expanded.npz`.

**Outputs:** `data/features_en_expanded.json`, `data/products_en_expanded.json`, `data/similarity_expanded.npz`.

**Dependencies:** see `requirements.txt`. The Qwen model (~1.2 GB) is downloaded on the first run.

## 0. Preprocess the expanded catalog

Selects the default English locale, includes variant-override content, decodes HTML entities, trims text, and aggregates repeated titles. If a product's titles and descriptions have different lengths, the descriptions are kept as unpaired instead of guessing the pairing with `zip`.

In [1]:
import html
import json
from pathlib import Path

data_dir = Path("data")
if not (data_dir / "expanded_catalog.json").is_file():
    data_dir = Path("CTFDS-Project/data")

catalog = json.loads((data_dir / "expanded_catalog.json").read_text(encoding="utf-8"))
default_locale = catalog.get("default_locale", "en-GB")
products = catalog["products"]

features_path = data_dir / "features_en_expanded.json"
products_path = data_dir / "products_en_expanded.json"

blocks = []
for raw_product in products:
    product_type = raw_product["data"]["classification"]["product_type"]
    product_id = raw_product["id"]
    content = (raw_product.get("data") or {}).get("content") or {}
    blocks.append((product_id, product_type, content.get(default_locale, {})))

    for variant in raw_product.get("variants") or []:
        variant_content = ((variant.get("overrides") or {}).get("content") or {})
        blocks.append((product_id, product_type, variant_content.get(default_locale, {})))

attribute_records = {}
product_records = {}
unpaired_descriptions = []

for product_id, product_type, content in blocks:
    titles = [html.unescape(title).strip() for title in content.get("feature_titles") or []]
    descriptions = [html.unescape(description).strip() for description in content.get("features") or []]

    product = product_records.setdefault(
        product_id,
        {"product_id": product_id, "product_type": product_type, "titles": set()},
    )
    product["titles"].update(titles)

    for title in titles:
        attribute_records.setdefault(title, {"title": title, "descriptions": []})

    if len(titles) != len(descriptions):
        unpaired_descriptions.extend(descriptions)
        continue

    for title, description in zip(titles, descriptions):
        attribute_records[title]["descriptions"].append(description)

for product in product_records.values():
    product["titles"] = sorted(product["titles"])

features_path.write_text(
    json.dumps(
        {"attribute_records": list(attribute_records.values()), "unpaired_descriptions": unpaired_descriptions},
        ensure_ascii=False,
        indent=1,
    ),
    encoding="utf-8",
)
products_path.write_text(
    json.dumps(list(product_records.values()), ensure_ascii=False, indent=1),
    encoding="utf-8",
)

print(f"Wrote {len(attribute_records)} unique feature titles to {features_path}")
print(f"Kept {len(unpaired_descriptions)} descriptions that could not be paired safely")
print(f"Wrote {len(product_records)} products to {products_path}")

Wrote 664 unique feature titles to data/features_en_expanded.json
Kept 0 descriptions that could not be paired safely
Wrote 349 products to data/products_en_expanded.json


In [2]:
import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
from rapidfuzz import fuzz, process, utils
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# weights of the three similarity signals
W_SEMANTIC, W_TFIDF, W_FUZZY = 0.50, 0.25, 0.25
W_TITLE = 0.50
PENALTY = 0.30
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
EMBEDDING_PROMPT = "Instruct: Identify product features that describe the same property\nQuery: "

W1127 23:16:00.580000 71118 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 1. Load the feature records

Loads the titles and their descriptions from `data/features_en_expanded.json`. The unpaired descriptions are kept separately and only used for the TF-IDF word weights.

In [3]:
feature_data = json.loads(features_path.read_text(encoding="utf8"))
attribute_records = feature_data["attribute_records"]
unpaired_descriptions = feature_data["unpaired_descriptions"]

attributes = [record["title"] for record in attribute_records]
descriptions = [record["descriptions"] for record in attribute_records]

print(len(attributes), "titles,", sum(bool(d) for d in descriptions), "with descriptions,",
      len(unpaired_descriptions), "unpaired descriptions")

664 titles, 664 with descriptions, 0 unpaired descriptions


## 2. TF-IDF similarity of the descriptions

Each title's descriptions are joined into one document. TF-IDF gives rare words more weight than words used everywhere. The word weights are learned from all descriptions but only titles get a row in the matrix. Titles without descriptions get 0 here and rely on the other two signals.

In [4]:
#here, each title has its document
documents = [" ".join(texts) for texts in descriptions]

# word weights are learned from all descriptions
tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True)
tfidf.fit([d for d in documents if d] + unpaired_descriptions)

# rows are normalised, so the dot product is the cosine similarity
vectors = tfidf.transform(documents)
tfidf_similarity = (vectors @ vectors.T).toarray()
print("TF-IDF similarity matrix:", tfidf_similarity.shape)

TF-IDF similarity matrix: (664, 664)


## 3. Semantic title-and-description embeddings

An embedding is a vector (a point in a high-dimensional meaning space) produced from a piece of text. Texts with related meanings tend to have vectors pointing in similar directions, even when they use different words. This step creates one semantic vector for each feature title; it does not cluster the titles yet.

For every title, the model embeds the title text and each distinct, aligned description. If a title has several descriptions, their vectors are averaged using the number of times each description occurs in the data. The title vector and this description-context vector are then blended equally (`W_TITLE = 0.50`) and normalized. Later, the cosine similarity between two resulting vectors gives their raw semantic similarity.

**Why cosine similarity?** The vectors are normalized to length 1, so their dot product equals cosine similarity: `cos(theta) = (u · v) / (||u|| ||v||) = u · v`. Cosine compares vector direction (semantic orientation) rather than length. “Raw” means this score is calculated before the median-based calibration later in this notebook.

**Other possible comparisons:** Euclidean distance (`||u - v||`) is a natural alternative. For unit-length vectors, it gives the same ranking as cosine because `||u - v||^2 = 2(1 - cosine_similarity)`; only the distance scale changes. Manhattan distance (`sum(abs(u - v))`) is another option, but it measures separation differently and may rank pairs differently, so it should be checked against reviewed similar/dissimilar title pairs. A plain dot product is also possible, but without normalization it is sensitive to vector length; with these normalized vectors it is identical to cosine similarity.

**Example — “Light insulation”:** this title has five description entries but three distinct texts. “Delivering warmth and exceptional breathability for comfort during transitional rides” occurs three times; the other two descriptions occur once each. The description-context vector is therefore conceptually `(3 × vector(description 1) + vector(description 2) + vector(description 3)) / 5`. That context is blended 50/50 with the embedding of the title “Light insulation”. The repeated description pulls the context toward its meaning, but does not change the title’s own embedding.

If a title has no aligned description, its semantic vector is just its title embedding—not a zero vector. Descriptions that could not safely be matched to titles are excluded from this step. They are used only when fitting TF-IDF in the next section. The public model may need to download the first time; no access token or login is required.

In [5]:
import torch

embedding_model = SentenceTransformer(MODEL_NAME, model_kwargs={"dtype": torch.float32})
title_vectors = embedding_model.encode(
    attributes,
    prompt=EMBEDDING_PROMPT,
    normalize_embeddings=True,
    show_progress_bar=True,
)

all_descriptions = sorted({text for texts in descriptions for text in texts})
description_embedding_vectors = embedding_model.encode(
    all_descriptions,
    prompt=EMBEDDING_PROMPT,
    normalize_embeddings=True,
    show_progress_bar=True,
)
description_vector_by_text = {
    text: description_embedding_vectors[index]
    for index, text in enumerate(all_descriptions)
}

attribute_vectors = []
for title_vector, texts in zip(title_vectors, descriptions):
    if texts:
        description_counts = Counter(texts)
        weighted_description_vector = np.average(
            [description_vector_by_text[text] for text in description_counts],
            axis=0,
            weights=list(description_counts.values()),
        )
        vector = W_TITLE * title_vector + (1 - W_TITLE) * weighted_description_vector
        vector = vector / np.linalg.norm(vector)
    else:
        vector = title_vector
    attribute_vectors.append(vector)

attribute_vectors = np.asarray(attribute_vectors)
raw_semantic = attribute_vectors @ attribute_vectors.T
baseline_similarity = np.median(raw_semantic[np.triu_indices(len(attributes), k=1)])
if baseline_similarity >= 1:
    raise ValueError("Semantic similarities cannot be calibrated: median baseline is 1.")

# Calibrate model-specific cosine baselines to 0; identical vectors remain at 1.
semantic = np.clip(
    (raw_semantic - baseline_similarity) / (1 - baseline_similarity),
    0,
    1,
)
print(f"Semantic matrix ready; uncalibrated median baseline: {baseline_similarity:.3f}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Semantic matrix ready; uncalibrated median baseline: 0.565


## 4. Blend the three similarities

Fuzzy matching compares the title wording itself, ignoring word order ("Cycling Gloves" = "Gloves Cycling"). The final similarity is a weighted average: 50% semantic, 25% TF-IDF and 25% fuzzy. We blend the similarity matrices instead of combining the vectors, because embeddings and TF-IDF vectors live in different spaces.

In [6]:
assert np.isclose(W_SEMANTIC + W_TFIDF + W_FUZZY, 1), "weights must sum to 1"

fuzzy = process.cdist(
    attributes,
    attributes,
    scorer=fuzz.token_sort_ratio,
    processor=utils.default_process,
) / 100

#rescaling fuzzy the same way as semantic
fuzzy_baseline = np.median(fuzzy[np.triu_indices(len(attributes), k=1)])
fuzzy = np.clip((fuzzy - fuzzy_baseline) / (1 - fuzzy_baseline), 0, 1)

similarity = (
    W_SEMANTIC * semantic
    + W_TFIDF *  tfidf_similarity
    + W_FUZZY * fuzzy
)
# A missing description gives no IDF evidence, but every item has zero distance to itself.
np.fill_diagonal(similarity, 1.0)
print(f"Blended similarity matrix: {similarity.shape}")

Blended similarity matrix: (664, 664)


## 5. Penalty for conflicting concepts

Embeddings see "Deep winter protection" and "Windproof protection" as similar, but they describe different properties (insulation vs. blocking wind). Pairs where one title is only about winter/cold and the other only about wind lose 0.30 similarity . Titles that mention both are not penalised.


> **TODO:** this rule was carried over from an earlier notebook so we need to check it. We need to compare clustering results with and without the penalty once the evaluation exists

In [7]:
CONCEPTS = {
    "winter": r"\b(winter|cold|freez\w*|frost\w*)\b",
    "windproof": r"\b(wind\w*)\b",
}
CONFLICTS = [("winter", "windproof")]

concepts_by_attribute = [
    {
        name
        for name, pattern in CONCEPTS.items()
        if re.search(pattern, title, re.IGNORECASE)
    }
    for title in attributes
]

penalty_matrix = np.zeros_like(similarity)
for first_concept, second_concept in CONFLICTS:
    first_only = np.array(
        [first_concept in concepts and second_concept not in concepts for concepts in concepts_by_attribute]
    )
    second_only = np.array(
        [second_concept in concepts and first_concept not in concepts for concepts in concepts_by_attribute]
    )
    penalty_matrix[np.ix_(first_only, second_only)] = PENALTY
    penalty_matrix[np.ix_(second_only, first_only)] = PENALTY

similarity = np.clip(similarity - penalty_matrix, 0, 1)
print(f"Conflicting attribute pairs penalized: {int(np.count_nonzero(penalty_matrix) // 2)}")

Conflicting attribute pairs penalized: 414


In [8]:

# Save the shared similarity matrix for all clustering notebooks
output_path = data_dir / "similarity_expanded.npz"
output_path.parent.mkdir(exist_ok=True)

np.savez_compressed(
    output_path,
    similarity=similarity,                 # n x n, values in [0, 1]
    titles=np.array(attributes),           # row/column order of the matrix
)
print(f"Saved {similarity.shape} similarity matrix to {output_path}")

Saved (664, 664) similarity matrix to data/similarity_expanded.npz
